# Spark Exercises 06 — Nested & Semi-structured Data

This is where Spark leaves a flat pandas/Polars table behind. Real event data arrives as **nested JSON**: a `device` object, a `geo` object, and an `items` **array** of products. Spark reads this natively into `struct` and `array` columns and gives you first-class tools — dot access, `explode`, `from_json` — to work with it without ever flattening to messy strings.

**Data file:** `data/events.jsonl` — 2,500 clickstream events (JSON Lines).

---

> **Solutions version** — try it yourself first from `Exercises.ipynb`.

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 3.5.1


### 2. Read the JSON Lines file with `spark.read.json("data/events.jsonl")` into `events`, then `printSchema()`. Notice the **nested structs** (`device`, `geo`) and the **array** (`items`).

In [2]:
events = spark.read.json("data/events.jsonl")
events.printSchema()

root
 |-- device: struct (nullable = true)
 |    |-- browser: string (nullable = true)
 |    |-- is_mobile: boolean (nullable = true)
 |    |-- os: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- geo: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- country: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |    |    |-- sku: string (nullable = true)
 |-- search_term: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)



### 3. Show 2 events **vertically** so you can see the nested structure rendered.

In [3]:
events.show(2, vertical=True, truncate=False)

-RECORD 0-----------------------------------------------------------------
 device      | {Safari, true, Windows}                                    
 event_id    | EV-0000001                                                 
 event_ts    | 2024-06-20T09:04:50                                        
 event_type  | add_to_cart                                                
 geo         | {Sydney, AU}                                               
 items       | [{119.96, 4, EL502}, {41.65, 2, SH901}, {99.28, 1, EL500}] 
 search_term | NULL                                                       
 session_id  | S-0793                                                     
 user_id     | CUST-0053                                                  
-RECORD 1-----------------------------------------------------------------
 device      | {Firefox, true, Windows}                                   
 event_id    | EV-0000002                                                 
 event_ts    | 2024-06-20

### 4. **Dot access.** Reach into the structs: select `event_id`, `event_type`, `device.os`, `device.is_mobile`, `geo.country`. Show 8 rows.

In [4]:
events.select("event_id", "event_type", "device.os",
              "device.is_mobile", "geo.country").show(8)

+----------+-----------+-------+---------+-------+
|  event_id| event_type|     os|is_mobile|country|
+----------+-----------+-------+---------+-------+
|EV-0000001|add_to_cart|Windows|     true|     AU|
|EV-0000002|add_to_cart|Windows|     true|     US|
|EV-0000003|  page_view|  Linux|     true|     SG|
|EV-0000004|add_to_cart|  macOS|     true|     UK|
|EV-0000005|  page_view|  Linux|    false|     US|
|EV-0000006|  page_view|Windows|     true|     DE|
|EV-0000007|     search|    iOS|    false|     DE|
|EV-0000008|     search|  Linux|     true|     BR|
+----------+-----------+-------+---------+-------+
only showing top 8 rows



### 5. Count events **per operating system** (`device.os`), most common first.

In [5]:
events.groupBy("device.os").count().orderBy(F.col("count").desc()).show()

+-------+-----+
|     os|count|
+-------+-----+
|Android|  513|
|Windows|  507|
|  Linux|  506|
|    iOS|  487|
|  macOS|  487|
+-------+-----+



### 6. **Flatten a struct.** Use `select("device.*")` to expand all of `device`'s fields into top-level columns. Show 5 rows.

In [6]:
events.select("device.*").show(5)

+-------+---------+-------+
|browser|is_mobile|     os|
+-------+---------+-------+
| Safari|     true|Windows|
|Firefox|     true|Windows|
| Chrome|     true|  Linux|
| Chrome|     true|  macOS|
|   Edge|    false|  Linux|
+-------+---------+-------+
only showing top 5 rows



### 7. **Filter on nested fields.** Keep only events from mobile devices using the `Chrome` browser. How many are there?

In [7]:
mobile_chrome = events.filter(
    (F.col("device.is_mobile") == True) & (F.col("device.browser") == "Chrome")
)
mobile_chrome.count()

359

### 8. The `items` array is only present for `add_to_cart` / `purchase` events. Add a column `n_items = size(items)` and show it for 8 events (events without items show `-1`).

In [8]:
events.withColumn("n_items", F.size("items")) \
      .select("event_id", "event_type", "n_items").show(8)

+----------+-----------+-------+
|  event_id| event_type|n_items|
+----------+-----------+-------+
|EV-0000001|add_to_cart|      3|
|EV-0000002|add_to_cart|      1|
|EV-0000003|  page_view|     -1|
|EV-0000004|add_to_cart|      1|
|EV-0000005|  page_view|     -1|
|EV-0000006|  page_view|     -1|
|EV-0000007|     search|     -1|
|EV-0000008|     search|     -1|
+----------+-----------+-------+
only showing top 8 rows



### 9. **Explode the array.** Take only `purchase` events, then `explode` `items` so each product in a basket becomes its **own row**. Select `event_id`, `user_id`, `item.sku`, `item.qty`, `item.price`. Show 10.

In [9]:
purchases = events.filter(F.col("event_type") == "purchase")
lines = purchases.select(
    "event_id", "user_id",
    F.explode("items").alias("item")
)
lines.select("event_id", "user_id", "item.sku", "item.qty", "item.price").show(10)

+----------+---------+-----+---+-----+
|  event_id|  user_id|  sku|qty|price|
+----------+---------+-----+---+-----+
|EV-0000009|CUST-0368|PT010|  3|51.24|
|EV-0000009|CUST-0368|SH901|  4|75.26|
|EV-0000009|CUST-0368|BK300|  3|67.45|
|EV-0000019|CUST-0240|SH900|  4|  9.4|
|EV-0000019|CUST-0240|PT044|  2|  7.8|
|EV-0000023|CUST-0169|EL500|  3|45.88|
|EV-0000023|CUST-0169|MG777|  3| 4.82|
|EV-0000034|CUST-0119|PB024|  2|65.65|
|EV-0000034|CUST-0119|SH901|  3|83.89|
|EV-0000034|CUST-0119|NB050|  1|68.37|
+----------+---------+-----+---+-----+
only showing top 10 rows



### 10. From the exploded `lines`, compute total **purchase revenue** (`sum(qty * price)`), rounded to 2 decimals.

In [10]:
lines.select(F.round(F.sum(F.col("item.qty") * F.col("item.price")), 2).alias("revenue")).show()

+--------+
| revenue|
+--------+
|66180.51|
+--------+



### 11. Which **5 SKUs** were purchased most often (count of exploded item rows)?

In [11]:
lines.groupBy(F.col("item.sku").alias("sku")) \
     .count().orderBy(F.col("count").desc()).show(5)

+-----+-----+
|  sku|count|
+-----+-----+
|PB024|   29|
|SH900|   27|
|BK302|   26|
|NB050|   25|
|EL502|   25|
+-----+-----+
only showing top 5 rows



### 12. **`posexplode`** also gives the position within the array. On `purchases` use `posexplode(items)` to get `pos` and `item`, and show the first item (`pos == 0`) of 8 baskets.

In [12]:
purchases.select("event_id", F.posexplode("items").alias("pos", "item")) \
         .filter(F.col("pos") == 0) \
         .select("event_id", "item.sku", "item.qty").show(8)

+----------+-----+---+
|  event_id|  sku|qty|
+----------+-----+---+
|EV-0000009|PT010|  3|
|EV-0000019|SH900|  4|
|EV-0000023|EL500|  3|
|EV-0000034|PB024|  2|
|EV-0000057|EL501|  4|
|EV-0000063|PB200|  2|
|EV-0000066|SH900|  4|
|EV-0000069|EL502|  1|
+----------+-----+---+
only showing top 8 rows



### 13. **Build and parse JSON.** Turn the `device` struct back into a JSON string with `F.to_json`, then parse it back into a struct with `F.from_json` using an explicit schema. Show `os` and `browser` from the re-parsed struct.

In [13]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType

device_schema = StructType([
    StructField("os", StringType()),
    StructField("browser", StringType()),
    StructField("is_mobile", BooleanType()),
])
events.withColumn("device_json", F.to_json("device")) \
      .withColumn("parsed", F.from_json("device_json", device_schema)) \
      .select("device_json", "parsed.os", "parsed.browser").show(5, truncate=False)

+-----------------------------------------------------+-------+-------+
|device_json                                          |os     |browser|
+-----------------------------------------------------+-------+-------+
|{"browser":"Safari","is_mobile":true,"os":"Windows"} |Windows|Safari |
|{"browser":"Firefox","is_mobile":true,"os":"Windows"}|Windows|Firefox|
|{"browser":"Chrome","is_mobile":true,"os":"Linux"}   |Linux  |Chrome |
|{"browser":"Chrome","is_mobile":true,"os":"macOS"}   |macOS  |Chrome |
|{"browser":"Edge","is_mobile":false,"os":"Linux"}    |Linux  |Edge   |
+-----------------------------------------------------+-------+-------+
only showing top 5 rows

